# VehicleMakeNet — Baseline vs Fine-tuned Evaluation on YMAD

**Модели:**
- Baseline: VehicleMakeNet ResNet-18 (NVIDIA NGC `pruned_onnx_v1.1.0`, 20 классов)
- Fine-tuned: `vmn_vmmrdb_ymad_105c` (105 классов, VMMRDB + YMAD)

**Датасет:** YMAD (Yandex Moscow Automotive Dataset) — стратифицированная выборка 2 000 авто / ~10 432 изображений  
**Инференс:** ONNX Runtime CUDA (GPU на remote kernel)

## Цель
Измерить Top-1 / Top-3 accuracy и per-class F1 VehicleMakeNet на YMAD:
- Baseline оценивается только на брендах из 20 классов модели
- Fine-tuned оценивается на всех брендах из своих 105 классов

## Гипотеза
Fine-tuned модель достигает Top-1 > 0.90 и Top-3 > 0.97 на пересечении брендов YMAD.

## Preprocessing (NVIDIA TAO VehicleMakeNet)
```
RGB, resize 224×224, (pixel − mean) × scale
mean = [123.675, 116.28, 103.53]  # ImageNet RGB×255
scale = 1/255
```

In [3]:
%pip install numpy matplotlib opencv-python-headless onnxruntime-gpu tqdm

  Using cached matplotlib-3.10.8-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.7 MB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (60.4 MB)
  Using cached onnxruntime_gpu-1.23.2-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (300.5 MB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached contourpy-1.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (325 kB)
  Using cached fonttools-4.62.1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.9 MB)
  Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
  Using cached kiwisolver-1.5.0-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl (1.6 MB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl (26 kB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl (46 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl (86 kB)
  Using cached mpmath-1.3.0-py3-none-any

In [7]:
!pip install onnxruntime-gpu

  Using cached onnxruntime_gpu-1.23.2-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (300.5 MB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl (26 kB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl (46 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl (86 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device: '/usr/local/lib/python3.10/dist-packages/mpmath'



In [8]:
import json
import os
import subprocess
import time
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import onnxruntime as ort
from tqdm import tqdm


def find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists() or (p / "models").exists():
            return p
    return Path.cwd()


ROOT = find_project_root()
print("ROOT:", ROOT)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
CFG = {
    "baseline": {
        "model_dir": ROOT / "models" / "vehiclemakenet_pruned_onnx_v1.1.0",
        "onnx_file": "resnet18_pruned.onnx",
        "labels_file": "labels.txt",
        "n_classes": 20,
        "input_name": "input_1:0",
        "output_name": "predictions/Softmax:0",
        "output_is_logits": False,
    },
    "finetuned": {
        "model_dir": ROOT / "models" / "vmn_vmmrdb_ymad_105c",
        "onnx_file": "model.onnx",
        "labels_file": "labels.txt",
        "n_classes": 105,
        "input_name": "input",
        "output_name": "logits",
        "output_is_logits": True,
    },
    "data": {
        "images_index": ROOT / "data" / "external" / "ymad_cars" / "images_index.jsonl",
        "images_dir": ROOT / "data" / "external" / "ymad_cars" / "images",
    },
    "preprocess": {
        "input_size": (224, 224),
        "mean_rgb": np.array([123.675, 116.28, 103.53], dtype=np.float32),
        "scale": 1.0 / 255.0,
    },
    "eval": {
        "batch_size": 128,
        "providers": ["CUDAExecutionProvider", "CPUExecutionProvider"],
        "target_top1": 0.90,
        "target_top3": 0.97,
        "max_cm_classes": 30,
        "error_examples_n": 12,
        "figures_dir": ROOT / "notebooks" / "vmn" / "vmn_eval",
    },
    "ngc": {
        "org": "nvidia",
        "team": "tao",
        "model": "vehiclemakenet",
        "version": "pruned_onnx_v1.1.0",
    },
}

CFG["eval"]["figures_dir"].mkdir(parents=True, exist_ok=True)
print("Config OK, figures →", CFG["eval"]["figures_dir"])

## Скачивание модели из NVIDIA NGC

Требует установленного NGC CLI (`ngc`) и валидного `NGC_API_KEY`.  
Если модель уже в `models/vehiclemakenet_pruned_onnx_v1.1.0/` — скачивание пропускается.

**Альтернатива без CLI:** скачать вручную с [NGC](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/tao/models/vehiclemakenet) и распаковать в `models/vehiclemakenet_pruned_onnx_v1.1.0/`.

In [ ]:
def download_vehiclemakenet_ngc(cfg: dict) -> bool:
    model_dir: Path = cfg["baseline"]["model_dir"]
    onnx_path = model_dir / cfg["baseline"]["onnx_file"]
    if onnx_path.exists():
        print(f"Model already present: {onnx_path}")
        return True

    ngc = cfg["ngc"]
    model_spec = f"{ngc['org']}/{ngc['team']}/{ngc['model']}:{ngc['version']}"
    dest = str(model_dir.parent)

    print(f"Downloading {model_spec} → {dest}")
    try:
        result = subprocess.run(
            ["ngc", "registry", "model", "download-version", model_spec, "--dest", dest],
            capture_output=True,
            text=True,
            timeout=600,
        )
        if result.returncode != 0:
            print("NGC stderr:", result.stderr[-2000:])
            raise RuntimeError(f"ngc exited with code {result.returncode}")
        print("NGC stdout:", result.stdout[-1000:])

        # ngc скачивает в подпапку вида vehiclemakenet_pruned_onnx_v1.1.0
        downloaded_dir = model_dir.parent / f"vehiclemakenet_{ngc['version']}"
        if downloaded_dir.exists() and not model_dir.exists():
            downloaded_dir.rename(model_dir)

        if onnx_path.exists():
            print("Download OK:", onnx_path)
            return True
        raise FileNotFoundError(f"Expected {onnx_path} after download")
    except FileNotFoundError as exc:
        if "ngc" in str(exc):
            print("NGC CLI not found. Trying wget fallback...")
            return _download_via_wget(cfg)
        raise


def _download_via_wget(cfg: dict) -> bool:
    """Fallback: скачать zip через NGC API (требует NGC_API_KEY в env)."""
    import urllib.request

    ngc = cfg["ngc"]
    api_key = os.environ.get("NGC_API_KEY", "")
    url = (
        f"https://api.ngc.nvidia.com/v2/models/{ngc['org']}/{ngc['team']}"
        f"/{ngc['model']}/versions/{ngc['version']}/zip"
    )
    model_dir: Path = cfg["baseline"]["model_dir"]
    zip_path = model_dir.parent / "vmn_baseline.zip"
    model_dir.mkdir(parents=True, exist_ok=True)

    headers = {"Authorization": f"ApiKey {api_key}"} if api_key else {}
    req = urllib.request.Request(url, headers=headers)
    print(f"GET {url}")
    with urllib.request.urlopen(req, timeout=600) as resp, zip_path.open("wb") as f:
        total = int(resp.headers.get("Content-Length", 0))
        downloaded = 0
        chunk = 1 << 20
        while True:
            data = resp.read(chunk)
            if not data:
                break
            f.write(data)
            downloaded += len(data)
            if total:
                print(f"\r  {downloaded/1e6:.1f}/{total/1e6:.1f} MB", end="")
    print()
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(model_dir.parent)
    zip_path.unlink()
    print("Extracted to", model_dir.parent)
    return (model_dir / cfg["baseline"]["onnx_file"]).exists()


download_vehiclemakenet_ngc(CFG)

In [ ]:
def load_labels(labels_path: Path) -> list[str]:
    text = labels_path.read_text().strip()
    if ";" in text:
        return [l.strip() for l in text.split(";") if l.strip()]
    return [l.strip() for l in text.splitlines() if l.strip()]


def build_session(model_cfg: dict, providers: list[str]) -> ort.InferenceSession:
    onnx_path = model_cfg["model_dir"] / model_cfg["onnx_file"]
    opts = ort.SessionOptions()
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(onnx_path), sess_options=opts, providers=providers)


sessions = {}
labels_map = {}

for name in ("baseline", "finetuned"):
    mcfg = CFG[name]
    labels = load_labels(mcfg["model_dir"] / mcfg["labels_file"])
    assert len(labels) == mcfg["n_classes"], f"{name}: expected {mcfg['n_classes']} labels, got {len(labels)}"
    labels_map[name] = labels

    sess = build_session(mcfg, CFG["eval"]["providers"])
    providers_used = sess.get_providers()
    sessions[name] = sess
    print(f"[{name}] providers={providers_used}  labels={len(labels)}: {labels[:5]}...")

print("\nBaseline labels:", labels_map["baseline"])
print("\nFine-tuned labels sample:", labels_map["finetuned"][:10])

## Датасет

Загружаем `images_index.jsonl`, дедуплицируем по `image_path` и фильтруем по классам каждой модели.

In [ ]:
def load_index(index_path: Path, images_dir: Path) -> list[dict]:
    seen = set()
    records = []
    with index_path.open() as f:
        for line in f:
            rec = json.loads(line)
            key = rec["image_path"]
            if key in seen:
                continue
            seen.add(key)
            rec["abs_path"] = images_dir / rec["image_path"]
            records.append(rec)
    return records


def filter_records(records: list[dict], label_set: set[str]) -> list[dict]:
    return [r for r in records if r["brand"] in label_set]


all_records = load_index(CFG["data"]["images_index"], CFG["data"]["images_dir"])
print(f"Total unique images: {len(all_records)}")

datasets = {}
for name in ("baseline", "finetuned"):
    label_set = set(labels_map[name])
    filtered = filter_records(all_records, label_set)
    existing = [r for r in filtered if r["abs_path"].exists()]
    datasets[name] = existing
    print(f"[{name}] brand coverage: {len(label_set)} classes | "
          f"matching records: {len(filtered)} | on disk: {len(existing)}")
    missing_on_disk = len(filtered) - len(existing)
    if missing_on_disk:
        print(f"  WARNING: {missing_on_disk} images missing on disk")

In [ ]:
def preprocess(img_bgr: np.ndarray, cfg: dict) -> np.ndarray:
    h, w = cfg["input_size"]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (w, h), interpolation=cv2.INTER_LINEAR)
    img_f = img_resized.astype(np.float32)
    img_f -= cfg["mean_rgb"]
    img_f *= cfg["scale"]
    return img_f.transpose(2, 0, 1)  # CHW


def run_inference(
    session: ort.InferenceSession,
    records: list[dict],
    model_cfg: dict,
    pre_cfg: dict,
    batch_size: int,
    labels: list[str],
) -> list[dict]:
    label2idx = {l: i for i, l in enumerate(labels)}
    input_name = model_cfg["input_name"]
    output_name = model_cfg["output_name"]
    is_logits = model_cfg["output_is_logits"]

    results = []
    n_batches = (len(records) + batch_size - 1) // batch_size

    for bi in tqdm(range(n_batches), desc=f"inference"):
        batch_recs = records[bi * batch_size: (bi + 1) * batch_size]
        imgs = []
        valid_recs = []
        for rec in batch_recs:
            img = cv2.imread(str(rec["abs_path"]))
            if img is None:
                continue
            imgs.append(preprocess(img, pre_cfg))
            valid_recs.append(rec)
        if not imgs:
            continue

        batch_arr = np.stack(imgs, axis=0)
        raw = session.run([output_name], {input_name: batch_arr})[0]

        if is_logits:
            exp = np.exp(raw - raw.max(axis=1, keepdims=True))
            probs = exp / exp.sum(axis=1, keepdims=True)
        else:
            probs = raw

        for rec, prob in zip(valid_recs, probs):
            gt_label = rec["brand"]
            gt_idx = label2idx.get(gt_label, -1)
            top3_idx = np.argsort(prob)[::-1][:3].tolist()
            results.append({
                "image_path": rec["image_path"],
                "gt_label": gt_label,
                "gt_idx": gt_idx,
                "pred_top1": labels[top3_idx[0]],
                "pred_top1_idx": top3_idx[0],
                "pred_top3_labels": [labels[i] for i in top3_idx],
                "pred_top3_idx": top3_idx,
                "top1_score": float(prob[top3_idx[0]]),
            })
    return results

In [ ]:
predictions = {}

for name in ("baseline", "finetuned"):
    t0 = time.time()
    preds = run_inference(
        session=sessions[name],
        records=datasets[name],
        model_cfg=CFG[name],
        pre_cfg=CFG["preprocess"],
        batch_size=CFG["eval"]["batch_size"],
        labels=labels_map[name],
    )
    elapsed = time.time() - t0
    predictions[name] = preds
    fps = len(preds) / elapsed if elapsed > 0 else 0
    print(f"[{name}] {len(preds)} images in {elapsed:.1f}s  ({fps:.0f} img/s)")

In [ ]:
def compute_metrics(preds: list[dict], labels: list[str]) -> dict:
    valid_preds = [p for p in preds if p["gt_idx"] >= 0]
    n = len(valid_preds)
    if n == 0:
        return {}

    top1_correct = sum(1 for p in valid_preds if p["gt_idx"] == p["pred_top1_idx"])
    top3_correct = sum(1 for p in valid_preds if p["gt_idx"] in p["pred_top3_idx"])

    label2idx = {l: i for i, l in enumerate(labels)}
    n_cls = len(labels)
    tp = np.zeros(n_cls, dtype=int)
    fp = np.zeros(n_cls, dtype=int)
    fn = np.zeros(n_cls, dtype=int)
    gt_counts = np.zeros(n_cls, dtype=int)

    for p in valid_preds:
        gt = p["gt_idx"]
        pred = p["pred_top1_idx"]
        gt_counts[gt] += 1
        if gt == pred:
            tp[pred] += 1
        else:
            fp[pred] += 1
            fn[gt] += 1

    precision = np.where(tp + fp > 0, tp / (tp + fp), 0.0)
    recall = np.where(tp + fn > 0, tp / (tp + fn), 0.0)
    f1 = np.where(precision + recall > 0, 2 * precision * recall / (precision + recall), 0.0)

    active_mask = gt_counts > 0
    macro_f1 = float(f1[active_mask].mean()) if active_mask.any() else 0.0
    weighted_f1 = float((f1 * gt_counts).sum() / gt_counts.sum()) if gt_counts.sum() > 0 else 0.0

    per_class = [
        {
            "label": labels[i],
            "n_gt": int(gt_counts[i]),
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
        }
        for i in range(n_cls)
        if gt_counts[i] > 0
    ]
    per_class.sort(key=lambda x: x["f1"], reverse=True)

    return {
        "n_samples": n,
        "top1": top1_correct / n,
        "top3": top3_correct / n,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "per_class": per_class,
    }


metrics = {}
for name in ("baseline", "finetuned"):
    m = compute_metrics(predictions[name], labels_map[name])
    metrics[name] = m
    target_top1 = CFG["eval"]["target_top1"]
    target_top3 = CFG["eval"]["target_top3"]
    ok1 = "✓" if m["top1"] >= target_top1 else "✗"
    ok3 = "✓" if m["top3"] >= target_top3 else "✗"
    print(
        f"[{name}]  n={m['n_samples']:>5}  "
        f"Top-1={m['top1']:.4f} {ok1}(>{target_top1})  "
        f"Top-3={m['top3']:.4f} {ok3}(>{target_top3})  "
        f"macro-F1={m['macro_f1']:.4f}  weighted-F1={m['weighted_f1']:.4f}"
    )

In [ ]:
def print_summary_table(metrics: dict, targets: dict):
    hdr = f"{'Model':<12} {'N':>6} {'Top-1':>8} {'Top-3':>8} {'Macro-F1':>10} {'W-F1':>8}"
    print("\n" + "=" * len(hdr))
    print("VehicleMakeNet — Summary Metrics")
    print("=" * len(hdr))
    print(hdr)
    print("-" * len(hdr))
    for name, m in metrics.items():
        ok1 = "✓" if m["top1"] >= targets["top1"] else "✗"
        ok3 = "✓" if m["top3"] >= targets["top3"] else "✗"
        print(
            f"{name:<12} {m['n_samples']:>6} "
            f"{m['top1']:>7.4f}{ok1} {m['top3']:>7.4f}{ok3} "
            f"{m['macro_f1']:>10.4f} {m['weighted_f1']:>8.4f}"
        )
    print("=" * len(hdr))


def plot_summary_table(metrics: dict, targets: dict, save_dir: Path):
    print_summary_table(metrics, targets)
    if not HAS_MATPLOTLIB:
        return

    rows = [
        {
            "Model": name,
            "N eval": m["n_samples"],
            "Top-1": f"{m['top1']:.4f}",
            "Top-3": f"{m['top3']:.4f}",
            "Macro-F1": f"{m['macro_f1']:.4f}",
            "W-F1": f"{m['weighted_f1']:.4f}",
        }
        for name, m in metrics.items()
    ]
    cols = list(rows[0].keys())
    cell_text = [[str(r[c]) for c in cols] for r in rows]

    fig, ax = plt.subplots(figsize=(10, len(rows) * 0.7 + 1))
    ax.axis("off")
    tbl = ax.table(cellText=cell_text, colLabels=cols, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1.2, 1.8)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor("#2c3e50")
            cell.set_text_props(color="white", weight="bold")
        elif col in (2, 3) and row > 0:
            key = "top1" if col == 2 else "top3"
            try:
                val = float(cell_text[row - 1][col])
                cell.set_facecolor("#2ecc71" if val >= targets[key] else "#e74c3c")
                cell.set_text_props(color="white")
            except ValueError:
                pass
    plt.title("VehicleMakeNet — Summary Metrics", pad=12, fontsize=13, weight="bold")
    path = save_dir / "summary_table.png"
    plt.savefig(path, bbox_inches="tight", dpi=150)
    plt.close()
    print("Saved:", path)


plot_summary_table(
    metrics,
    targets={"top1": CFG["eval"]["target_top1"], "top3": CFG["eval"]["target_top3"]},
    save_dir=CFG["eval"]["figures_dir"],
)

In [ ]:
def build_confusion_matrix(preds: list[dict], labels: list[str], max_classes: int) -> tuple[np.ndarray, list[str]]:
    gt_counts: dict[str, int] = {}
    for p in preds:
        gt = p["gt_label"]
        gt_counts[gt] = gt_counts.get(gt, 0) + 1

    top_labels = sorted(gt_counts, key=gt_counts.get, reverse=True)[:max_classes]
    top_set = set(top_labels)
    top_idx = {l: i for i, l in enumerate(top_labels)}
    n = len(top_labels)
    cm = np.zeros((n, n), dtype=int)

    for p in preds:
        gt = p["gt_label"]
        pred = p["pred_top1"]
        if gt not in top_set or pred not in top_set:
            continue
        cm[top_idx[gt], top_idx[pred]] += 1

    return cm, top_labels


def print_confusion_matrix_text(cm: np.ndarray, labels: list[str], top_confusions: int = 15):
    norm_cm = cm.astype(float)
    row_sums = norm_cm.sum(axis=1, keepdims=True)
    norm_cm = np.where(row_sums > 0, norm_cm / row_sums, 0)

    print(f"\n  Diagonal (recall per class):")
    for i, lbl in enumerate(labels):
        print(f"    {lbl:<20} {norm_cm[i,i]:.3f}  (n={cm[i].sum()})")

    print(f"\n  Top-{top_confusions} confusions (GT → Pred, normalized):")
    pairs = []
    for i in range(len(labels)):
        for j in range(len(labels)):
            if i != j and norm_cm[i, j] > 0:
                pairs.append((norm_cm[i, j], labels[i], labels[j], cm[i, j]))
    pairs.sort(reverse=True)
    for rate, gt, pred, count in pairs[:top_confusions]:
        print(f"    {gt:<20} → {pred:<20}  {rate:.3f}  (n={count})")


def plot_confusion_matrix(cm: np.ndarray, labels: list[str], title: str, save_path: Path):
    print_confusion_matrix_text(cm, labels)
    if not HAS_MATPLOTLIB:
        return

    n = len(labels)
    norm_cm = cm.astype(float)
    row_sums = norm_cm.sum(axis=1, keepdims=True)
    norm_cm = np.where(row_sums > 0, norm_cm / row_sums, 0)

    fig_size = max(10, n * 0.45)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    im = ax.imshow(norm_cm, cmap="Blues", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=90, fontsize=7 if n > 20 else 9)
    ax.set_yticklabels(labels, fontsize=7 if n > 20 else 9)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Ground Truth", fontsize=11)
    ax.set_title(title, fontsize=12, weight="bold")
    thresh = 0.5
    for i in range(n):
        for j in range(n):
            if norm_cm[i, j] > 0.01:
                ax.text(j, i, f"{norm_cm[i,j]:.2f}", ha="center", va="center",
                        fontsize=5 if n > 20 else 7,
                        color="white" if norm_cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.close()
    print("Saved:", save_path)


for name in ("baseline", "finetuned"):
    cm, cm_labels = build_confusion_matrix(
        predictions[name],
        labels_map[name],
        max_classes=CFG["eval"]["max_cm_classes"],
    )
    print(f"\n{'='*60}\n[{name}] Confusion Matrix (top-{len(cm_labels)} classes)\n{'='*60}")
    plot_confusion_matrix(
        cm,
        cm_labels,
        title=f"VehicleMakeNet [{name}] — Confusion Matrix (top-{len(cm_labels)} classes)",
        save_path=CFG["eval"]["figures_dir"] / f"confusion_matrix_{name}.png",
    )

In [ ]:
def plot_per_class_f1(per_class: list[dict], title: str, save_path: Path, top_n: int = 40):
    data = sorted(per_class, key=lambda x: x["n_gt"], reverse=True)[:top_n]
    data = sorted(data, key=lambda x: x["f1"], reverse=True)

    print(f"\n{title}")
    print(f"  {'Brand':<20} {'n_gt':>6} {'Prec':>7} {'Rec':>7} {'F1':>7}")
    print("  " + "-" * 50)
    for d in data:
        mark = "✓" if d["f1"] >= 0.9 else "~" if d["f1"] >= 0.7 else "✗"
        print(f"  {d['label']:<20} {d['n_gt']:>6} {d['precision']:>7.3f} {d['recall']:>7.3f} {d['f1']:>7.3f} {mark}")

    if not HAS_MATPLOTLIB:
        return

    labels = [d["label"] for d in data]
    f1_vals = [d["f1"] for d in data]
    n_gts = [d["n_gt"] for d in data]
    colors = ["#2ecc71" if v >= 0.9 else "#f39c12" if v >= 0.7 else "#e74c3c" for v in f1_vals]

    fig, ax = plt.subplots(figsize=(12, max(6, len(labels) * 0.32)))
    bars = ax.barh(labels, f1_vals, color=colors, edgecolor="white", linewidth=0.5)
    for bar, n_gt, f in zip(bars, n_gts, f1_vals):
        ax.text(min(f + 0.01, 0.97), bar.get_y() + bar.get_height() / 2,
                f"F1={f:.3f}  n={n_gt}", va="center", fontsize=7.5)
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("F1 Score", fontsize=11)
    ax.set_title(title, fontsize=12, weight="bold")
    ax.axvline(0.9, color="green", linestyle="--", linewidth=1, label="0.90 target")
    ax.axvline(0.7, color="orange", linestyle="--", linewidth=1, label="0.70")
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.close()
    print("Saved:", save_path)


for name in ("baseline", "finetuned"):
    plot_per_class_f1(
        metrics[name]["per_class"],
        title=f"VehicleMakeNet [{name}] — Per-class F1 (top-{CFG['eval']['max_cm_classes']} by support)",
        save_path=CFG["eval"]["figures_dir"] / f"per_class_f1_{name}.png",
    )

In [ ]:
def plot_baseline_vs_finetuned_f1(metrics: dict, save_path: Path):
    baseline_pc = {d["label"]: d["f1"] for d in metrics["baseline"]["per_class"]}
    finetuned_pc = {d["label"]: d["f1"] for d in metrics["finetuned"]["per_class"]}
    common = sorted(set(baseline_pc) & set(finetuned_pc))

    if not common:
        print("No common classes for comparison")
        return

    print(f"\nBaseline vs Fine-tuned — F1 on {len(common)} common brands:")
    print(f"  {'Brand':<20} {'Baseline':>10} {'Fine-tuned':>12} {'Delta':>8}")
    print("  " + "-" * 54)
    for c in common:
        b, f = baseline_pc[c], finetuned_pc[c]
        delta = f - b
        sign = "+" if delta >= 0 else ""
        print(f"  {c:<20} {b:>10.3f} {f:>12.3f} {sign}{delta:>7.3f}")

    if not HAS_MATPLOTLIB:
        return

    b_f1 = [baseline_pc[c] for c in common]
    f_f1 = [finetuned_pc[c] for c in common]
    x = np.arange(len(common))
    width = 0.38

    fig, ax = plt.subplots(figsize=(max(10, len(common) * 0.45), 5))
    ax.bar(x - width / 2, b_f1, width, label="Baseline (20 cls)", color="#3498db", alpha=0.85)
    ax.bar(x + width / 2, f_f1, width, label="Fine-tuned (105 cls)", color="#e67e22", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(common, rotation=60, ha="right", fontsize=8)
    ax.set_ylabel("F1 Score")
    ax.set_ylim(0, 1.1)
    ax.set_title("Baseline vs Fine-tuned — F1 per common brand", weight="bold")
    ax.axhline(0.9, color="green", linestyle="--", linewidth=1, label="target 0.90")
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.close()
    print("Saved:", save_path)


plot_baseline_vs_finetuned_f1(
    metrics,
    save_path=CFG["eval"]["figures_dir"] / "baseline_vs_finetuned_f1.png",
)

## Анализ ошибок

Для каждой модели — `N` примеров где Top-1 неверный, упорядоченные по уверенности модели (сложные случаи).

In [ ]:
def plot_error_examples(
    preds: list[dict],
    images_dir: Path,
    model_name: str,
    n: int,
    save_path: Path,
):
    errors = [p for p in preds if p["gt_idx"] != p["pred_top1_idx"]]
    errors.sort(key=lambda p: p["top1_score"], reverse=True)
    sample = errors[:n]

    total = len(preds)
    n_err = len(errors)
    print(f"\n[{model_name}] Errors: {n_err}/{total} ({100*n_err/max(1,total):.1f}%)")
    print(f"  Top-{len(sample)} highest-confidence mistakes:")
    print(f"  {'image':<30} {'GT':<20} {'Pred':<20} {'Score':>7}")
    print("  " + "-" * 82)
    for p in sample:
        img_name = Path(p["image_path"]).name
        print(f"  {img_name:<30} {p['gt_label']:<20} {p['pred_top1']:<20} {p['top1_score']:>7.3f}")

    if not HAS_MATPLOTLIB or not sample:
        return

    cols = 4
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.2))
    axes = np.array(axes).flatten()

    for ax, pred in zip(axes, sample):
        img_path = images_dir / Path(pred["image_path"]).name
        img = cv2.imread(str(img_path))
        if img is None:
            ax.axis("off")
            continue
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(
            f"GT: {pred['gt_label']}\nPred: {pred['pred_top1']} ({pred['top1_score']:.2f})",
            fontsize=8, color="red",
        )
        ax.axis("off")

    for ax in axes[len(sample):]:
        ax.axis("off")

    fig.suptitle(f"[{model_name}] Error examples (highest-confidence mistakes)", fontsize=11, weight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close()
    print("Saved:", save_path)


for name in ("baseline", "finetuned"):
    plot_error_examples(
        preds=predictions[name],
        images_dir=CFG["data"]["images_dir"],
        model_name=name,
        n=CFG["eval"]["error_examples_n"],
        save_path=CFG["eval"]["figures_dir"] / f"error_examples_{name}.png",
    )

In [ ]:
def print_worst_classes(per_class: list[dict], model_name: str, bottom_n: int = 10):
    worst = sorted(per_class, key=lambda x: x["f1"])[:bottom_n]
    print(f"\n[{model_name}] Worst {bottom_n} classes by F1:")
    print(f"  {'Brand':<20} {'n_gt':>6} {'Prec':>7} {'Rec':>7} {'F1':>7}")
    print("  " + "-" * 52)
    for d in worst:
        print(f"  {d['label']:<20} {d['n_gt']:>6} {d['precision']:>7.3f} {d['recall']:>7.3f} {d['f1']:>7.3f}")


for name in ("baseline", "finetuned"):
    print_worst_classes(metrics[name]["per_class"], model_name=name)

## Выводы

Заполняется после запуска ноутбука.

| Метрика | Baseline (20 cls) | Fine-tuned (105 cls) | Target |
|---------|-------------------|----------------------|--------|
| Top-1 | — | — | ≥ 0.90 |
| Top-3 | — | — | ≥ 0.97 |
| Macro-F1 | — | — | — |
| Weighted-F1 | — | — | — |

### Наблюдения
- Baseline охватывает только западные бренды (20 классов), большинство YMAD-авто (VAZ, GAZ и др.) не оценивается.
- Fine-tuned (105 cls) покрывает ~X% изображений YMAD (по пересечению брендов).
- Слабые классы (F1 < 0.7): ...
- Типичные ошибки: визуально похожие бренды (BMW↔Mercedes, Kia↔Hyundai и т.д.)

### Рекомендации
- Добавить hard-negative mining для визуально схожих пар брендов.
- Расширить датасет для редких брендов (< 100 примеров).
- Рассмотреть ensemble baseline + fine-tuned для западных брендов.